# Future Climate Data — Bias-Corrected GCM Processing

Builds the future-climate products the rest of the pipeline depends on, from
the 4 selected, bias-corrected GCMs (2 SSPs each: SSP2-4.5 and SSP5-8.5):

1. **Monthly climatology (near/mid/far future) & continuous annual series**
   per model x scenario, from `101_selected 4_GCMs/.../ssp{scenario}_bias_corrected.xlsx`.
2. **Per-station combined series** (all 8 model-scenario runs in one file per
   station) for both monthly climatology and the continuous annual series.
3. **Merged Tmax/Tmin file** per model x scenario (needed because the SWAT+
   climate format wants Tmax and Tmin together).
4. **Continuous daily SWAT+ climate input files** (2015–2100, gap-filled by
   linear interpolation) for every model x scenario combination — written to
   `New_Future_Climate/SWAT_input/{scenario}_{model}/`.

**Reads:** `101_selected 4_GCMs/{ppt,tmax,tmin}/GCM/{model}/ssp{scenario}_bias_corrected.xlsx`,
plus the historical baseline files produced by `1_historical_climate_updated.ipynb`
(`Historical/Monthly/historical_monthly_average_{pcp,tmax,tmin}.csv`).

**Consumed by:** `3_future_climate_annual.ipynb`, `4_climate_change_station_level_plots.ipynb`
(via the per-station files from step 2), and the SWAT+ model runs (via step 4).

In [ ]:
import os

import pandas as pd

from climate_aggregation import monthly_climatology, yearly_aggregate

## Config — model/scenario mapping

Maps each raw GCM name to the climate-archetype label. SSP2-4.5 and
SSP5-8.5 use two different GCMs for "Hot-Dry" and "Cool-Wet" (the other two
archetypes share a GCM across both SSPs). - OUTPUTS from envelope based GCM selection

In [ ]:
models_245 = {
    'UKESM1-0-LL': 'Hot-Wet',
    'CNRM-CM6-1': 'Hot-Dry',
    'INM-CM5-0': 'Cool-Wet',
    'GFDL-ESM4': 'Cool-Dry'
}

models_585 = {
    'UKESM1-0-LL': 'Hot-Wet',
    'IPSL-CM6A-LR': 'Hot-Dry',
    'INM-CM4-8': 'Cool-Wet',
    'GFDL-ESM4': 'Cool-Dry'
}

models_245_only = list(models_245.keys())
models_585_only = list(models_585.keys())

variables = ['ppt', 'tmax', 'tmin']
scenarios = ['245', '585']

# The three future periods this project evaluates separately, each a full
# Jan-Dec span so monthly climatology and calendar months line up cleanly
FUTURE_PERIODS = {
    'near': ('2026-01-01', '2050-12-31'),
    'mid': ('2051-01-01', '2075-12-31'),
    'far': ('2076-01-01', '2100-12-31'),
}

models_245_only, models_585_only


## Step 1 — Monthly climatology & continuous annual series per model x scenario

In [ ]:
for variable in variables:
    for scenario in scenarios:
        models = models_245_only if scenario == '245' else models_585_only

        for model in models:
            future_bias_corrected = pd.read_excel(
                f'../101_selected 4_GCMs/{variable}/GCM/{model}/ssp{scenario}_bias_corrected.xlsx'
            )
            future_bias_corrected['Date'] = pd.to_datetime(
                future_bias_corrected['Date'], errors='coerce'
            )
            station_cols = [c for c in future_bias_corrected.columns if c not in ('Date', 'year', 'month', 'day')]
            how = 'sum' if variable == 'ppt' else 'mean'

            model_name = models_245[model] if scenario == '245' else models_585[model]
            output_dir = f'../All_DATA/Climate/New_Future_Climate/{variable}/ssp{scenario}/{model_name}'
            os.makedirs(output_dir, exist_ok=True)

            # -----------------------------
            # Monthly climatology (Jan-Dec average), computed separately
            # within each of the three future periods
            # -----------------------------
            for period, (start, end) in FUTURE_PERIODS.items():
                period_data = future_bias_corrected[
                    (future_bias_corrected['Date'] >= start) & (future_bias_corrected['Date'] <= end)
                ]
                monthly_climatology(period_data, 'Date', station_cols, how=how).to_csv(
                    f'{output_dir}/monthly_future_{period}.csv', index=False
                )

            # ======================================================
            # Continuous annual data (2026 onward)
            # ======================================================
            annual_data = future_bias_corrected[future_bias_corrected['Date'] >= '2026-01-01']
            yearly_aggregate(annual_data, 'Date', station_cols, how=how).round(2).to_csv(
                f'{output_dir}/annual_future_2026_2100.csv', index=False
            )


## Step 2 — Historical baseline & station lists

Historical monthly climatology comes from `1_historical_climate_updated.ipynb`.
Station lists and the 4 climate-archetype folder names are derived from what's
actually on disk rather than hardcoded, so they can't silently drift out of
sync with step 1's output.

In [ ]:
historical_monthly_pcp = pd.read_csv('../All_DATA/Climate/Historical/Monthly/historical_monthly_average_pcp.csv')
historical_monthly_tmax = pd.read_csv('../All_DATA/Climate/Historical/Monthly/historical_monthly_average_tmax.csv')
historical_monthly_tmin = pd.read_csv('../All_DATA/Climate/Historical/Monthly/historical_monthly_average_tmin.csv')

pcp_stations = historical_monthly_pcp.columns[1:].tolist()
temp_stations = historical_monthly_tmax.columns[1:].tolist()

model_archetypes = os.listdir('../All_DATA/Climate/New_Future_Climate/ppt/ssp245')

pcp_stations, temp_stations, model_archetypes


## Step 3 — Combine per-station series across all 8 model x scenario runs

In [ ]:
historical_data = [historical_monthly_pcp, historical_monthly_tmax, historical_monthly_tmin]

for variable, data in zip(variables, historical_data):
    stations = pcp_stations if variable == 'ppt' else temp_stations

    for station in stations:

        # =====================================================
        # Monthly climatology: historical + all 8 future runs, one row per month
        # =====================================================
        station_df = pd.DataFrame()
        station_df['Month'] = data['Month']
        station_df['historical'] = data[station]

        for scenario in ['245', '585']:
            for model in model_archetypes:
                for period in ['near', 'mid', 'far']:
                    future_monthly = pd.read_csv(
                        f'../All_DATA/Climate/New_Future_Climate/{variable}/ssp{scenario}/{model}/monthly_future_{period}.csv'
                    )
                    station_df[f'{scenario}_{model}_{period}'] = future_monthly[f'st_{station}']

        save_dir = f'../All_DATA/Climate/New_Future_Climate/{variable}/station_data_monthly'
        os.makedirs(save_dir, exist_ok=True)
        station_df.to_csv(f'{save_dir}/{station}_monthly_future.csv', index=False)

        # =====================================================
        # Annual continuous time series: one row per year, one column per run
        # =====================================================
        annual_df = None

        for scenario in ['245', '585']:
            for model in model_archetypes:
                annual_data = pd.read_csv(
                    f'../All_DATA/Climate/New_Future_Climate/{variable}/ssp{scenario}/{model}/annual_future_2026_2100.csv'
                )
                if annual_df is None:
                    annual_df = pd.DataFrame({'Year': annual_data['Year']})
                annual_df[f'{scenario}_{model}'] = annual_data[f'st_{station}']

        save_dir = f'../All_DATA/Climate/New_Future_Climate/{variable}/station_data_annual'
        os.makedirs(save_dir, exist_ok=True)
        annual_df.to_csv(f'{save_dir}/{station}_annual_future.csv', index=False)


## Step 4 — Merge Tmax/Tmin bias-corrected series

The SWAT+ climate format needs Tmax and Tmin together in one file per day,
so this merges the two bias-corrected series (per model x scenario) before
the daily SWAT+ export in step 5.

In [ ]:
for scenario in scenarios:
    models = models_245_only if scenario == '245' else models_585_only
    for model in models:
        future_bias_corrected_tmax = pd.read_excel(f'../101_selected 4_GCMs/tmax/GCM/{model}/ssp{scenario}_bias_corrected.xlsx')
        future_bias_corrected_tmin = pd.read_excel(f'../101_selected 4_GCMs/tmin/GCM/{model}/ssp{scenario}_bias_corrected.xlsx')
        future_bias_corrected_temp = pd.merge(
            future_bias_corrected_tmax, future_bias_corrected_tmin,
            on=['Date', 'year', 'month', 'day'], suffixes=('_tmax', '_tmin'),
        )
        future_bias_corrected_temp.to_csv(
            f'../101_selected 4_GCMs/temp/future_bias_corrected_temp_{model}_ssp{scenario}.csv', index=False
        )


## Step 5 — SWAT+ climate input files (continuous daily series, 2015–2100)

Reindexes each model x scenario series onto a single continuous daily date
range and linearly interpolates any gaps that introduces, then writes the
SWAT+-ready `.txt` files (one per variable per station) with the `20150101`
start-date header SWAT+ expects.

Writes to `New_Future_Climate/SWAT_input/`

In [ ]:
swat_variables = ['ppt', 'temp']
full_date_range = pd.date_range(start='2015-01-01', end='2100-12-31', freq='D')

for variable in swat_variables:
    stations = pcp_stations if variable == 'ppt' else temp_stations

    for scenario in scenarios:
        models = models_245_only if scenario == '245' else models_585_only

        for model in models:
            if variable == 'ppt':
                future_daily_data = pd.read_excel(f'../101_selected 4_GCMs/{variable}/GCM/{model}/ssp{scenario}_bias_corrected.xlsx')
            else:
                future_daily_data = pd.read_csv(f'../101_selected 4_GCMs/temp/future_bias_corrected_temp_{model}_ssp{scenario}.csv')

            # Reindex onto the full continuous date range, then linearly
            # interpolate any gaps that introduces (bias-corrected series
            # don't all start/end on the same date).
            future_daily_data['Date'] = pd.to_datetime(future_daily_data['Date'], errors='coerce')
            future_daily_data = future_daily_data.dropna(subset=['Date']).set_index('Date')
            future_daily_data = future_daily_data.reindex(full_date_range)
            future_daily_data.index.name = 'Date'
            future_daily_data = future_daily_data.interpolate(method='linear', limit_direction='both', axis=0)

            model_name = models_245[model] if scenario == '245' else models_585[model]
            working_dir = f'../All_DATA/Climate/New_Future_Climate/SWAT_input/{scenario}_{model_name}/'
            os.makedirs(working_dir, exist_ok=True)
            variable_name = 'pcp' if variable == 'ppt' else 'tmp'

            for station in stations:
                if variable == 'ppt':
                    col = f'st_{station}'
                    if col not in future_daily_data.columns:
                        print(f'Warning: column {col} not found for model {model} scenario {scenario}; skipping station {station}')
                        continue
                    station_data = future_daily_data[[col]]
                else:
                    station_data = future_daily_data[[f'st_{station}_tmax', f'st_{station}_tmin']]

                out_path = f'{working_dir}{variable_name}_{station}.txt'
                station_data.to_csv(out_path, index=False, header=False)
                with open(out_path, 'r+') as file:
                    content = file.read()
                    file.seek(0, 0)
                    file.write('20150101\n' + content)
